# RAG Demonstration

In [38]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import random
import re
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from typing import List
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json

In [39]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [40]:
df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/synq.csv")
queries = df['QUERY'].tolist()
passages = df['PASSAGE'].tolist()
# df.head(20)

### Example of a Query:

In [41]:
# print(queries[0])

### Example Passage:

In [42]:
# print(passages[0])

### For Patient:

In [43]:
print('Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]))

Subject ID: 22532


### Loading the vector DB, index and json

In [44]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/MEDRAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Loading the Trained Query Encoder

In [45]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained("bert-base-uncased")

query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/MEDRAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=512
        ) #.to("cuda")

    outputs = query_encoder(**inputs)
    cls_embeddings = outputs.last_hidden_state[:, 0] 

    embeddings.append(cls_embeddings.cpu())

    return torch.cat(embeddings, 0)

### Testing custom question

In [46]:
question = 'Subject ID: ' + str(df['SUBJECT_ID'].iloc[0]) + '\n'  +  "Does the patient have a history of tuberculosis?"
print(question)

Subject ID: 22532
Does the patient have a history of tuberculosis?


In [47]:
# Embedding Query
query_emb = encode_query([question]).detach().cpu().numpy()

In [48]:
# Matching for the top K=5 highest scores
K = 5
scores, ids = index.search(query_emb, K)

In [49]:
candidates = [metadata[i]["text"] for i in ids[0]]
for i in candidates:
    print(i)
    print("\n\n\n")

Subject ID: 14879
Type 2 diabetes, uncontrolled. 5. Anemia, acute blood loss. 6. Lymphoma. 7. Failure to thrive and deconditioning. DISCHARGE MEDICATIONS:
1. Tylenol 325-650 mg po q 4-6 h prn pain. 2. Pantoprazole 40 mg po qd. 3. Heparin subcu 5,000 U q 8 h. 4. Citalopram 20 mg po qd. 5. Mirtazapine 50 mg po q hs. 6. Epoetin Alfa 4,000 U 2 x week--Monday, Thursday. 7. Colace 100 mg po bid--hold for loose stools. 8.




Subject ID: 11018
- Continue long steroid taper at home (Prednisone 60mg X 7 days,
40mg X 7 days, 20 mg X7 days, 10mg X 7 days, off)
- Continue supplemental oxygen, albuterol and ipratropium nebs
- Continue MS contin and morphine liquid PRN for air hunger,
shortness of breath
- Continue lorazepam PRN for air hunger, shortness of breath,
anxiety
. # HIV: Down trending CD4 count, ?due to acute illness.




Subject ID: 68109
However, there is circumferential wall
thickening involving both ureters throughout its course. These
findings are overall suggestive of bilateral pyel

# Other infromation

### At this time:

In [50]:
# Synthetic Dataset length:
print(len(df))

28431


In [51]:
# Additional Synthetic Dataset Length:
add_df = pd.read_csv("/work/mbouthil/projects/research_project/MEDRAG/synthetic_data/add_synq.csv")
print(len(add_df))

79825


In [52]:
# add_df.head(20)